# Занятие 5. Промышленный краулинг: Scrapy и оркестрация в n8n — демонстрационный блокнот

**Курс:** Автоматизированный сбор данных из интернета (код + no-code)
**Формат:** демонстрация + база для лабораторной работы

Все прогоны выполняются на тренировочной площадке [books.toscrape.com](https://books.toscrape.com/) (1000 книг, 50 страниц пагинации, 50 категорий) — она создана специально для обучения краулингу.

Пауки запускаются как отдельные процессы через `scrapy runspider`. Это сознательное решение: Scrapy работает на собственном цикле событий, который конфликтует с циклом Jupyter, а запуск в подпроцессе одновременно решает проблему и показывает студентам реальный способ эксплуатации краулера.

### Содержание + замечания по блокноту

| Блок | На что обратить внимание |
|---|---|
| 0. Подготовка | версии, рабочий каталог, вспомогательный запуск |
| 1. Селекторы Scrapy | `css`, `xpath`, `.get()/.getall()`, `attrib` |
| 2. Первый паук: пагинация | `start_urls`, `parse`, `response.follow`, статистика |
| 3. Двухуровневый обход | список → карточка, `cb_kwargs` |
| 4. Item Pipeline | валидация, очистка, дедупликация, SQLite, `DropItem` в статистике |
| 5. Вежливость и скорость | `DOWNLOAD_DELAY`, AutoThrottle, замер |
| 6. Кэш и возобновление | `HTTPCACHE`, `JOBDIR` |
| 7. n8n-эквивалент | таблица соответствия компонентов |

 ⚠️В блоке 2 обратите внимание на итоговую статистику Scrapy: именно она, а не «скрипт вроде отработал», является отчётом о качестве сбора. К концу занятия статистика должна читаться так же свободно, как логи компилятора.

## 0. Подготовка

Устанавливается одной командой: `pip install scrapy`. Рабочие файлы складываем в отдельный каталог, чтобы блокнот не засорял рабочую папку.

In [ ]:
pip install scrapy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 379.8/379.8 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 492.7/492.7 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 89.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 270.6/270.6 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.5/107.5 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.6/74.6 kB 7.1 MB/s eta 0:00:00


In [ ]:
import json, os, re, shutil, sqlite3, subprocess, sys, time, textwrap
from pathlib import Path
import pandas as pd
import requests
import scrapy
from scrapy.selector import Selector

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 30)

WORK = Path("lecture5_demo").resolve()
WORK.mkdir(exist_ok=True)
os.chdir(WORK)
print("Scrapy:", scrapy.__version__, "| Python:", sys.version.split()[0])
print("Рабочий каталог:", WORK)

BASE = "https://books.toscrape.com/"

def write(name: str, text: str) -> Path:
    p = WORK / name
    p.write_text(textwrap.dedent(text).lstrip(), encoding="utf-8")
    return p

def run_spider(file: str, out: str | None = None, settings: dict | None = None,
               extra: list[str] | None = None, timeout: int = 420):
    """Запускает паука отдельным процессом и возвращает (время, stdout+stderr)."""
    cmd = [sys.executable, "-m", "scrapy", "runspider", file]
    if out:
        Path(out).unlink(missing_ok=True)
        cmd += ["-O", out]
    for k, v in (settings or {}).items():
        cmd += ["-s", f"{k}={v}"]
    cmd += extra or []
    t0 = time.perf_counter()
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout, cwd=WORK)
    return round(time.perf_counter() - t0, 2), r.stdout + r.stderr

def show_stats(path="stats.json", keys=None):
    """Печатает ключевые метрики прогона из файла статистики."""
    st = json.loads(Path(path).read_text(encoding="utf-8"))
    keys = keys or ["downloader/request_count", "downloader/response_status_count/200",
                    "item_scraped_count", "item_dropped_count", "elapsed_time_seconds",
                    "downloader/response_bytes", "retry/count", "log_count/WARNING"]
    rows = [{"метрика": k, "значение": st.get(k)} for k in keys if k in st]
    extra = {k: v for k, v in st.items() if k.startswith(("item_dropped_reasons", "dropped/"))}
    rows += [{"метрика": k, "значение": v} for k, v in extra.items()]
    return pd.DataFrame(rows)

print("Вспомогательные функции готовы.")

Scrapy: 2.19.0 | Python: 3.13.16
Рабочий каталог: /content/lecture5_demo
Вспомогательные функции готовы.


## 1. Селекторы Scrapy

Scrapy использует свою реализацию селекторов на базе parsel: `response.css()` и `response.xpath()` возвращают `SelectorList`, из которого извлекают `.get()` (первое значение или `None`) либо `.getall()` (список). Отличие от BeautifulSoup — единый API для CSS и XPath и возможность цепочек.

Чтобы не запускать паука, разбираем скачанный HTML напрямую через `Selector` — именно так работает `scrapy shell`.

In [ ]:
resp = requests.get(BASE, timeout=20, headers={"User-Agent": "course-crawler/1.0"})
resp.encoding = "utf-8"          # сайт не объявляет кодировку корректно: £ иначе превращается в «Â£»
html = resp.text
sel = Selector(text=html)        # экземпляр  Selector класса

# CSS:
print("Книг на странице :", len(sel.css("article.product_pod")))
print("Первый заголовок :", sel.css("article.product_pod h3 a::attr(title)").get())
print("Первая цена      :", sel.css("article.product_pod .price_color::text").get())
print("Наличие (текст)  :", sel.css("article.product_pod .instock.availability::text").getall()[1].strip())

# XPath: то же самое, плюс работа с текстом узла
print("\nXPath-вариант цены:", sel.xpath("//article[@class='product_pod'][1]//p[@class='price_color']/text()").get())
print("Ссылка 'next'     :", sel.css("li.next a::attr(href)").get())

# Атрибуты через .attrib и рейтинг, закодированный в класс
first = sel.css("article.product_pod")[0]
print("Класс рейтинга    :", first.css("p.star-rating").attrib["class"])
print("Категории в меню  :", len(sel.css(".side_categories ul li ul li a")))

rows = []
for b in sel.css("article.product_pod"):
    rows.append({
        "title": b.css("h3 a::attr(title)").get(),
        "price": b.css(".price_color::text").get(),
        "rating": b.css("p.star-rating").attrib["class"].replace("star-rating ", ""),
        "url": b.css("h3 a::attr(href)").get(),
    })
display(pd.DataFrame(rows).head(5))

Книг на странице : 20
Первый заголовок : A Light in the Attic
Первая цена      : £51.77
Наличие (текст)  : In stock

XPath-вариант цены: £51.77
Ссылка 'next'     : catalogue/page-2.html
Класс рейтинга    : star-rating Three
Категории в меню  : 50


,title,price,rating,url
0,A Light in the Attic,£51.77,Three,catalogue/a-light-in-the-attic_1000/index.html
1,Tipping the Velvet,£53.74,One,catalogue/tipping-the-velvet_999/index.html
2,Soumission,£50.10,One,catalogue/soumission_998/index.html
3,Sharp Objects,£47.82,Four,catalogue/sharp-objects_997/index.html
4,Sapiens: A Brief History of Humankind,£54.23,Five,catalogue/sapiens-a-brief-history-of-humankind...


**Что подчеркнуть:** `.get()` возвращает `None`, а не бросает исключение — это защищает краулер от падения на нетипичной странице, но требует проверки полноты данных в пайплайне. Отладка селекторов делается командой `scrapy shell "https://books.toscrape.com/"` без запуска полного паука.

## 2. Первый паук: пагинация и статистика

Минимальный паук: `name`, `start_urls`, `parse`. Переход на следующую страницу — через `response.follow`, который сам разрешает относительные ссылки. В учебном запуске обязателен предохранитель `CLOSESPIDER_PAGECOUNT`.

Метод `closed()` сохраняет статистику прогона в файл — дальше мы читаем её как отчёт о качестве сбора.

In [ ]:
%%writefile spider_basic.py
import json, scrapy


class BooksSpider(scrapy.Spider):
    name = "books_basic"
    start_urls = ["https://books.toscrape.com/"]

    custom_settings = {
        "USER_AGENT": "course-crawler/1.0 (educational use)",
        "ROBOTSTXT_OBEY": True,             # обязательное требование курса
        "CONCURRENT_REQUESTS_PER_DOMAIN": 2,
        "DOWNLOAD_DELAY": 0.25,
        "DOWNLOAD_TIMEOUT": 20,
        "CLOSESPIDER_PAGECOUNT": 6,         # предохранитель учебного запуска
        "LOG_LEVEL": "INFO",
        "FEED_EXPORT_ENCODING": "utf-8",
    }

    def parse(self, response):
        """Страница списка: извлекаем карточки и переходим на следующую страницу."""
        for book in response.css("article.product_pod"):
            yield {
                "title": book.css("h3 a::attr(title)").get(),
                "price_raw": book.css(".price_color::text").get(),
                "rating_raw": book.css("p.star-rating").attrib.get("class"),
                "url": response.urljoin(book.css("h3 a::attr(href)").get()),
                "page": response.url,
            }
        next_page = response.css("li.next a::attr(href)").get()
        if next_page:                                    # пагинация
            yield response.follow(next_page, callback=self.parse)

    def closed(self, reason):
        """Отчёт о прогоне: статистика Scrapy сохраняется в файл."""
        stats = {k: (v if isinstance(v, (int, float)) else str(v))
                 for k, v in self.crawler.stats.get_stats().items()}
        stats["close_reason"] = reason
        with open("stats.json", "w", encoding="utf-8") as f:
            json.dump(stats, f, ensure_ascii=False, indent=2)


Writing spider_basic.py


↓ запускаем созданного ранее паука

In [ ]:
secs, log = run_spider("spider_basic.py", out="books_basic.json")
items = json.loads(Path("books_basic.json").read_text(encoding="utf-8"))
print(f"Прогон занял {secs} c, собрано записей: {len(items)}")
display(pd.DataFrame(items).head(3))
display(show_stats())

Прогон занял 3.4 c, собрано записей: 100


,title,price_raw,rating_raw,url,page
0,A Light in the Attic,£51.77,star-rating Three,https://books.toscrape.com/catalogue/a-light-i...,https://books.toscrape.com/
1,Tipping the Velvet,£53.74,star-rating One,https://books.toscrape.com/catalogue/tipping-t...,https://books.toscrape.com/
2,Soumission,£50.10,star-rating One,https://books.toscrape.com/catalogue/soumissio...,https://books.toscrape.com/


,метрика,значение
0,downloader/request_count,6
1,downloader/response_status_count/200,5
2,item_scraped_count,100
3,downloader/response_bytes,28737


### Как читать статистику прогона

`downloader/request_count` — сколько запросов ушло (включая robots.txt); `downloader/response_status_count/*` — распределение кодов ответа; `item_scraped_count` — сколько записей отдал паук; `item_dropped_count` — сколько отбросил пайплайн; `elapsed_time_seconds` — длительность; `retry/count` — число повторов. Требование курса: в отчёте по лабораторной работе эти значения приводятся обязательно — они доказывают, что сбор прошёл корректно и вежливо.

In [ ]:
interesting = [l for l in log.splitlines()
               if "Crawled" in l or "Closing spider" in l or "Dumping" in l]
print("Фрагмент лога Scrapy:")
for l in interesting[:8]:
    print("  ", l[:150])

Фрагмент лога Scrapy:
   2026-10-06 14:47:30 [scrapy.extensions.logstats] INFO: Crawled 0 pages (at 0 pages/min), scraped 0 items (at 0 items/min)
   2026-10-06 14:47:32 [scrapy.core.engine] INFO: Closing spider (closespider_pagecount)
   2026-10-06 14:47:32 [scrapy.statscollectors] INFO: Dumping Scrapy stats:


## 3. Двухуровневый обход: список → карточка

Реальная задача почти всегда двухуровневая: со страницы списка берём ссылки, а детали читаем на карточке товара. Контекст со страницы списка передаём через `cb_kwargs` — это чище, чем `meta`, и явно виден в подписи колбэка.

In [ ]:
%%writefile spider_detail.py
import json, scrapy


class BooksDetailSpider(scrapy.Spider):
    name = "books_detail"
    start_urls = ["https://books.toscrape.com/"]

    custom_settings = {
        "USER_AGENT": "course-crawler/1.0 (educational use)",
        "ROBOTSTXT_OBEY": True,
        "CONCURRENT_REQUESTS_PER_DOMAIN": 4,
        "DOWNLOAD_DELAY": 0.2,
        "AUTOTHROTTLE_ENABLED": True,
        "CLOSESPIDER_ITEMCOUNT": 30,        # предохранитель
        "DEPTH_LIMIT": 3,
        "LOG_LEVEL": "INFO",
    }

    def parse(self, response):
        """Уровень 1: список. Ссылки на карточки + контекст через cb_kwargs."""
        for book in response.css("article.product_pod"):
            href = book.css("h3 a::attr(href)").get()
            yield response.follow(
                href, callback=self.parse_book,
                cb_kwargs={"list_price": book.css(".price_color::text").get(),
                           "list_page": response.url},
            )
        next_page = response.css("li.next a::attr(href)").get()
        if next_page:
            yield response.follow(next_page, callback=self.parse)

    def parse_book(self, response, list_price, list_page):
        """Уровень 2: карточка товара. Таблица характеристик → словарь."""
        table = {row.css("th::text").get(): row.css("td::text").get()
                 for row in response.css("table.table-striped tr")}
        yield {
            "title": response.css("h1::text").get(),
            "price_raw": response.css(".price_color::text").get(),
            "list_price_raw": list_price,
            "upc": table.get("UPC"),
            "availability_raw": table.get("Availability"),
            "reviews": table.get("Number of reviews"),
            "category": response.css(".breadcrumb li:nth-child(3) a::text").get(),
            "url": response.url,
            "list_page": list_page,
        }

    def closed(self, reason):
        stats = {k: (v if isinstance(v, (int, float)) else str(v))
                 for k, v in self.crawler.stats.get_stats().items()}
        stats["close_reason"] = reason
        with open("stats.json", "w", encoding="utf-8") as f:
            json.dump(stats, f, ensure_ascii=False, indent=2)


Writing spider_detail.py


In [ ]:
secs, log = run_spider("spider_detail.py", out="books_detail.json")
det = json.loads(Path("books_detail.json").read_text(encoding="utf-8"))
print(f"Прогон занял {secs} c, карточек собрано: {len(det)}")

ddf = pd.DataFrame(det)
display(ddf[["title", "price_raw", "upc", "availability_raw", "category", "reviews"]].head(5))
print("\nПорядок карточек не совпадает с порядком на сайте: Scrapy обрабатывает запросы")
print("асинхронно, поэтому логика паука не должна опираться на порядок ответов.")
display(show_stats())

Прогон занял 26.47 c, карточек собрано: 45


,title,price_raw,upc,availability_raw,category,reviews
0,It's Only the Himalayas,£45.17,a22124811bfa8350,In stock (19 available),Travel,0
1,Libertarianism for Beginners,£51.33,a18a4f574854aced,In stock (19 available),Politics,0
2,Mesaerion: The Best Science Fiction Stories 18...,£37.59,e30f54cea9b38190,In stock (19 available),Science Fiction,0
3,Olio,£23.88,feb7cc7701ecf901,In stock (19 available),Poetry,0
4,Our Band Could Be Your Life: Scenes from the A...,£57.25,deda3e61b9514b83,In stock (19 available),Music,0



Порядок карточек не совпадает с порядком на сайте: Scrapy обрабатывает запросы
асинхронно, поэтому логика паука не должна опираться на порядок ответов.


,метрика,значение
0,downloader/request_count,50
1,downloader/response_status_count/200,49
2,item_scraped_count,45
3,downloader/response_bytes,197909


## 4. Item Pipeline: валидация, очистка, дедупликация, хранение

Пайплайн — цепочка классов, выполняемых по возрастанию номера в `ITEM_PIPELINES`. Каждое звено решает одну задачу:

| № | Класс | Задача |
|---|---|---|
| 100 | `ValidationPipeline` | обязательные поля; иначе `DropItem` |
| 200 | `CleanPipeline` | цена → float, наличие → int, рейтинг → число, обрезка пробелов |
| 300 | `DedupPipeline` | дубли по UPC (бизнес-ключ) |
| 400 | `SQLitePipeline` | пакетная запись в БД, `open_spider` / `close_spider` |

Отброшенные записи обязательно попадают в статистику через `stats.inc_value` — иначе молчаливая потеря данных останется незамеченной.

Два технических момента, которые стоит проговорить: в актуальных версиях Scrapy методы пайплайна вызываются **без аргумента `spider`**, а доступ к статистике получают через `from_crawler` (в примере это вынесено в `StatsMixin`); путь к классу в `ITEM_PIPELINES` указывается от имени модуля файла (`spider_pipeline.CleanPipeline`), а в полноценном проекте — `myproject.pipelines.CleanPipeline`.

In [ ]:
%%writefile spider_pipeline.py
import json, re, sqlite3, scrapy
from scrapy.exceptions import DropItem

RATING = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}


class StatsMixin:
    """Доступ к статистике прогона по современному API (без аргумента spider)."""

    @classmethod
    def from_crawler(cls, crawler):
        return cls(crawler)

    def __init__(self, crawler):
        self.crawler = crawler

    def count(self, key, n=1):
        self.crawler.stats.inc_value(key, n)


class ValidationPipeline(StatsMixin):
    """100 — обязательные поля. Отсутствуют → запись отбрасывается."""
    REQUIRED = ("title", "price_raw", "upc")

    def process_item(self, item):
        missing = [f for f in self.REQUIRED if not item.get(f)]
        if missing:
            self.count("dropped/missing_fields")
            raise DropItem("нет обязательных полей: %s" % missing)
        return item


class CleanPipeline(StatsMixin):
    """200 — нормализация типов и текста."""

    def process_item(self, item):
        item["title"] = (item.get("title") or "").strip()
        price = re.sub(r"[^\d.]", "", item.get("price_raw") or "")
        try:
            item["price"] = float(price)
        except ValueError:
            self.count("dropped/bad_price")
            raise DropItem("цена не разбирается: %r" % item.get("price_raw"))
        m = re.search(r"(\d+)", item.get("availability_raw") or "")
        item["stock"] = int(m.group(1)) if m else 0
        rating_raw = (item.get("rating_raw") or "").split()
        item["rating"] = RATING.get(rating_raw[-1]) if rating_raw else None
        item["reviews"] = int(item.get("reviews") or 0)
        for k in ("price_raw", "availability_raw", "rating_raw"):
            item.pop(k, None)
        return item


class DedupPipeline(StatsMixin):
    """300 — дедупликация по бизнес-ключу (UPC), а не по URL."""

    def open_spider(self):
        self.seen = set()

    def process_item(self, item):
        key = item["upc"]
        if key in self.seen:
            self.count("dropped/duplicate")
            raise DropItem("дубль UPC %s" % key)
        self.seen.add(key)
        return item


class SQLitePipeline(StatsMixin):
    """400 — пакетная запись в SQLite; соединение живёт весь прогон."""
    BATCH = 20

    def open_spider(self):
        self.conn = sqlite3.connect("books.db")
        self.conn.execute(
            "CREATE TABLE IF NOT EXISTS books ("
            " upc TEXT PRIMARY KEY, title TEXT NOT NULL, price REAL,"
            " stock INTEGER, rating INTEGER, reviews INTEGER,"
            " category TEXT, url TEXT)"
        )
        self.conn.commit()
        self.buffer = []

    def process_item(self, item):
        self.buffer.append((item["upc"], item["title"], item["price"], item["stock"],
                            item["rating"], item["reviews"], item.get("category"),
                            item["url"]))
        if len(self.buffer) >= self.BATCH:
            self._flush()
        return item

    def _flush(self):
        if not self.buffer:
            return
        self.conn.executemany(
            "INSERT OR REPLACE INTO books VALUES (?,?,?,?,?,?,?,?)", self.buffer)
        self.conn.commit()
        self.count("db/rows_written", len(self.buffer))
        self.buffer.clear()

    def close_spider(self):
        self._flush()
        self.conn.close()


class BooksPipelineSpider(scrapy.Spider):
    name = "books_pipeline"
    start_urls = ["https://books.toscrape.com/"]

    custom_settings = {
        "USER_AGENT": "course-crawler/1.0 (educational use)",
        "ROBOTSTXT_OBEY": True,
        "CONCURRENT_REQUESTS_PER_DOMAIN": 4,
        "AUTOTHROTTLE_ENABLED": True,
        "CLOSESPIDER_ITEMCOUNT": 40,
        "LOG_LEVEL": "WARNING",
        "ITEM_PIPELINES": {                  # порядок = возрастание номера
            "spider_pipeline.ValidationPipeline": 100,
            "spider_pipeline.CleanPipeline": 200,
            "spider_pipeline.DedupPipeline": 300,
            "spider_pipeline.SQLitePipeline": 400,
        },
    }

    def parse(self, response):
        for book in response.css("article.product_pod"):
            yield response.follow(book.css("h3 a::attr(href)").get(),
                                  callback=self.parse_book)
        nxt = response.css("li.next a::attr(href)").get()
        if nxt:
            yield response.follow(nxt, callback=self.parse)

    def parse_book(self, response):
        table = {r.css("th::text").get(): r.css("td::text").get()
                 for r in response.css("table.table-striped tr")}
        yield {
            "title": response.css("h1::text").get(),
            "price_raw": response.css(".price_color::text").get(),
            "rating_raw": response.css("p.star-rating").attrib.get("class"),
            "availability_raw": table.get("Availability"),
            "reviews": table.get("Number of reviews"),
            "upc": table.get("UPC"),
            "category": response.css(".breadcrumb li:nth-child(3) a::text").get(),
            "url": response.url,
        }

    def closed(self, reason):
        stats = {k: (v if isinstance(v, (int, float)) else str(v))
                 for k, v in self.crawler.stats.get_stats().items()}
        stats["close_reason"] = reason
        with open("stats.json", "w", encoding="utf-8") as f:
            json.dump(stats, f, ensure_ascii=False, indent=2)


Writing spider_pipeline.py


In [ ]:
Path("books.db").unlink(missing_ok=True)
secs, log = run_spider("spider_pipeline.py", out="books_clean.json")
clean = json.loads(Path("books_clean.json").read_text(encoding="utf-8"))
print(f"Прогон занял {secs} c, записей после пайплайна: {len(clean)}")

cdf = pd.DataFrame(clean)
display(cdf[["title", "price", "stock", "rating", "reviews", "category"]].head(5))
print("\nТипы полей после очистки:")
print(cdf[["price", "stock", "rating", "reviews"]].dtypes.to_string())
display(show_stats(keys=["downloader/request_count", "item_scraped_count",
                         "item_dropped_count", "db/rows_written",
                         "elapsed_time_seconds", "retry/count"]))

Прогон занял 21.56 c, записей после пайплайна: 55


,title,price,stock,rating,reviews,category
0,It's Only the Himalayas,45.17,19,2,0,Travel
1,Libertarianism for Beginners,51.33,19,2,0,Politics
2,Mesaerion: The Best Science Fiction Stories 18...,37.59,19,1,0,Science Fiction
3,Olio,23.88,19,1,0,Poetry
4,Our Band Could Be Your Life: Scenes from the A...,57.25,19,3,0,Music



Типы полей после очистки:
price      float64
stock        int64
rating       int64
reviews      int64


,метрика,значение
0,downloader/request_count,61
1,item_scraped_count,55
2,db/rows_written,55


### Проверка результата в базе

Пайплайн записал данные в SQLite — дальше начинается аналитика, то есть цель сбора достигнута. Полезно показать это сразу: студенты видят смысл всей конструкции.

In [ ]:
conn = sqlite3.connect("books.db")
print("Строк в таблице books:", conn.execute("SELECT COUNT(*) FROM books").fetchone()[0])
print("Уникальных UPC       :", conn.execute("SELECT COUNT(DISTINCT upc) FROM books").fetchone()[0])

q = """
SELECT category,
       COUNT(*)              AS книг,
       ROUND(AVG(price), 2)  AS средняя_цена,
       ROUND(AVG(rating), 2) AS средний_рейтинг,
       SUM(stock)            AS остаток
FROM books
GROUP BY category
ORDER BY книг DESC
LIMIT 8
"""
display(pd.read_sql_query(q, conn))

full = pd.read_sql_query("SELECT * FROM books", conn)
print("\nПолнота полей, % (тот же приём, что на занятии 3):")
print((full.notna().mean() * 100).round(1).to_string())
conn.close()

Строк в таблице books: 55
Уникальных UPC       : 55


,category,книг,средняя_цена,средний_рейтинг,остаток
0,Default,8,27.63,2.75,147
1,Nonfiction,6,34.72,3.50,100
2,Sequential Art,5,37.90,3.00,83
3,Poetry,5,28.92,2.40,90
4,Young Adult,3,28.44,3.00,54
5,Childrens,3,49.22,1.67,53
6,Thriller,2,38.02,4.50,36
7,Mystery,2,38.06,2.50,34



Полнота полей, % (тот же приём, что на занятии 3):
upc         100.0
title       100.0
price       100.0
stock       100.0
rating      100.0
reviews     100.0
category    100.0
url         100.0


## 5. Вежливость против скорости

Сравниваем два режима на одинаковом объёме: агрессивный (высокий параллелизм, без задержки) и вежливый (AutoThrottle, ограничение запросов на домен). Студенты должны увидеть, что цена вежливости — единицы секунд, а выигрыш — отсутствие нагрузки на сайт и риска блокировки.

AutoThrottle измеряет время ответа сервера и сам подбирает задержку: чем медленнее отвечает сайт, тем осторожнее ведёт себя краулер.

In [ ]:
BASE_SETTINGS = {"CLOSESPIDER_ITEMCOUNT": 40, "LOG_LEVEL": "ERROR"}

fast = {**BASE_SETTINGS, "CONCURRENT_REQUESTS": 16, "CONCURRENT_REQUESTS_PER_DOMAIN": 16,
        "DOWNLOAD_DELAY": 0, "AUTOTHROTTLE_ENABLED": False}
polite = {**BASE_SETTINGS, "CONCURRENT_REQUESTS": 8, "CONCURRENT_REQUESTS_PER_DOMAIN": 2,
          "DOWNLOAD_DELAY": 0.5, "AUTOTHROTTLE_ENABLED": True,
          "AUTOTHROTTLE_TARGET_CONCURRENCY": 1.0}

rows = []
for label, st, fname in (("агрессивный", fast, "cmp_fast.json"),
                         ("вежливый (AutoThrottle)", polite, "cmp_polite.json")):
    secs, _ = run_spider("spider_basic.py", out=fname, settings=st)
    n = len(json.loads(Path(fname).read_text(encoding="utf-8")))
    s = json.loads(Path("stats.json").read_text(encoding="utf-8"))
    rows.append({"режим": label, "секунд": secs, "записей": n,
                 "запросов": s.get("downloader/request_count"),
                 "байт": s.get("downloader/response_bytes"),
                 "повторов": s.get("retry/count", 0)})

comp = pd.DataFrame(rows)
comp["сек/запись"] = (comp["секунд"] / comp["записей"]).round(3)
display(comp)

ratio = comp.loc[1, "секунд"] / max(comp.loc[0, "секунд"], 1e-9)
print(f"Вежливый режим медленнее в {ratio:.1f} раза — это и есть цена вежливости.")
print("Что получаем взамен: нагрузка на сайт снижается кратно, растёт шанс дойти до конца")
print("прогона без блокировки, а на больших объёмах AutoThrottle сам подстраивается под сервер.")
print("Правило курса: агрессивный режим допустим только на собственном стенде.")

,режим,секунд,записей,запросов,байт,повторов,сек/запись
0,агрессивный,1.94,60,4,17094,0,0.032
1,вежливый (AutoThrottle),14.05,60,4,17126,0,0.234


Вежливый режим медленнее в 7.2 раза — это и есть цена вежливости.
Что получаем взамен: нагрузка на сайт снижается кратно, растёт шанс дойти до конца
прогона без блокировки, а на больших объёмах AutoThrottle сам подстраивается под сервер.
Правило курса: агрессивный режим допустим только на собственном стенде.


## 6. Кэш ответов и возобновление прогона

**HTTP-кэш** (`HTTPCACHE_ENABLED`) сохраняет ответы на диск: при отладке селекторов повторные запуски не обращаются к сайту вообще. Это одновременно ускорение и этичное поведение.

In [ ]:
shutil.rmtree(".scrapy", ignore_errors=True)
cache = {"CLOSESPIDER_PAGECOUNT": 5, "LOG_LEVEL": "ERROR", "DOWNLOAD_DELAY": 0.25,
         "HTTPCACHE_ENABLED": True, "HTTPCACHE_EXPIRATION_SECS": 3600}

t1, _ = run_spider("spider_basic.py", out="cache1.json", settings=cache)
s1 = json.loads(Path("stats.json").read_text(encoding="utf-8"))
t2, _ = run_spider("spider_basic.py", out="cache2.json", settings=cache)
s2 = json.loads(Path("stats.json").read_text(encoding="utf-8"))

display(pd.DataFrame([
    {"прогон": "первый (кэш пустой)", "секунд": t1,
     "запросов": s1.get("downloader/request_count"),
     "попаданий в кэш": s1.get("httpcache/hit", 0),
     "записано в кэш": s1.get("httpcache/store", 0)},
    {"прогон": "второй (из кэша)", "секунд": t2,
     "запросов": s2.get("downloader/request_count"),
     "попаданий в кэш": s2.get("httpcache/hit", 0),
     "записано в кэш": s2.get("httpcache/store", 0)},
]))
print("При отладке селекторов кэш экономит время и полностью снимает нагрузку с сайта.")

,прогон,секунд,запросов,попаданий в кэш,записано в кэш
0,первый (кэш пустой),2.58,5,0,5
1,второй (из кэша),1.26,5,5,0


При отладке селекторов кэш экономит время и полностью снимает нагрузку с сайта.


**Возобновление (`JOBDIR`).** Каталог состояния хранит очередь запросов и отпечатки уже посещённых URL, поэтому прогон можно остановить и продолжить с того же места — критично для многочасовых задач.

In [ ]:
shutil.rmtree("jobstate", ignore_errors=True)
job = {"JOBDIR": "jobstate", "CLOSESPIDER_ITEMCOUNT": 20,
       "LOG_LEVEL": "ERROR", "DOWNLOAD_DELAY": 0.2}

t1, _ = run_spider("spider_basic.py", out="job1.json", settings=job)
n1 = len(json.loads(Path("job1.json").read_text(encoding="utf-8")))
print(f"Первый прогон: {n1} записей за {t1} c")
print("Каталог состояния:", sorted(p.name for p in Path("jobstate").iterdir()))
seen = Path("jobstate/requests.seen")   # файл бинарный: отпечатки запросов
print("Отпечатков посещённых запросов:",
      len(seen.read_bytes().splitlines()) if seen.exists() else 0)

t2, _ = run_spider("spider_basic.py", out="job2.json", settings=job)
n2 = len(json.loads(Path("job2.json").read_text(encoding="utf-8")))
s = json.loads(Path("stats.json").read_text(encoding="utf-8"))
print(f"\nВторой прогон (возобновление): {n2} записей за {t2} c")
print("Отфильтровано как уже посещённые:", s.get("dupefilter/filtered", 0))
print("Очередь и дедупликация сохранились между запусками — это и есть возобновляемость.")

Первый прогон: 40 записей за 1.97 c
Каталог состояния: ['requests.queue', 'requests.seen', 'spider.state']
Отпечатков посещённых запросов: 3

Второй прогон (возобновление): 60 записей за 2.06 c
Отфильтровано как уже посещённые: 1
Очередь и дедупликация сохранились между запусками — это и есть возобновляемость.


## 7. Тот же конвейер в n8n

Scrapy и n8n решают пересекающиеся задачи, но с разных сторон: Scrapy — это движок обхода, n8n — оркестратор интеграций. Таблица соответствия, которую студенты переносят в конспект:

| Задача | Scrapy | n8n |
|---|---|---|
| Запуск по расписанию | внешний cron / Scrapyd | `Schedule Trigger` |
| Очередь и батчи | Scheduler с приоритетами | `Split Out` + `Loop Over Items` |
| Загрузка страницы | Downloader | `HTTP Request` (в т.ч. встроенная пагинация) |
| Парсинг | `response.css/xpath` | `HTML Extract` / `Code` |
| Валидация и фильтрация | `ValidationPipeline` | `Filter` / `IF` / `Code` |
| Очистка и типы | `CleanPipeline` | `Set` / `Code` |
| Дедупликация | `DedupPipeline`, dupefilter | `Remove Duplicates` |
| Контроль скорости | `DOWNLOAD_DELAY`, AutoThrottle | узел `Wait`, размер батча |
| Повторы | RetryMiddleware | Retry On Fail в настройках узла |
| Хранение | `SQLitePipeline`, Feed Exports | `Postgres` / `Google Sheets` / `S3` |
| Декомпозиция | модули и пауки | master + `Execute Sub-workflow` |
| Наблюдаемость | статистика прогона | журнал выполнений по узлам |
| Оповещения | свой код | ветка Error → Telegram/Email/Slack |

**Эскиз workflow для той же задачи:**
1. `Schedule Trigger` — ежедневно в 05:00.
2. `HTTP Request` — страница каталога; включена пагинация по ссылке «next» либо цикл по номерам страниц.
3. `HTML Extract` — массив карточек: заголовок, цена, ссылка, рейтинг.
4. `Split Out` — массив в отдельные items.
5. `Loop Over Items` (батч 10) → `Execute Sub-workflow` — подчинённый workflow получает карточку, делает `HTTP Request` на страницу товара и извлекает UPC, наличие, категорию.
6. `Wait` 1 с между батчами — аналог `DOWNLOAD_DELAY`.
7. `Remove Duplicates` по UPC → `Set` (типизация) → `Postgres` (UPSERT).
8. Ветка ошибок узлов → уведомление; в настройках узлов включён Retry On Fail (3 попытки).

**Где n8n сильнее:** расписание, интеграции и алерты из коробки; прозрачность для нетехнической команды; журнал выполнений с фактическими данными на каждом узле.

**Где Scrapy сильнее:** десятки тысяч страниц в одном прогоне, приоритеты и дедупликация очереди, возобновление после сбоя (`JOBDIR`), кэш, тестируемость и версионирование логики, стоимость при большом объёме.

**Вопрос для обсуждения (3 мин):** нужно ежедневно обходить 40 000 страниц каталога, складывать данные в PostgreSQL и присылать отчёт в мессенджер. Что выбираете? Ожидаемый разбор: сбор — Scrapy (очередь, дедупликация, возобновление, вежливость), оркестрация и отчёт — n8n, который запускает паука по API Scrapyd и по завершении читает из БД сводку. Чистый n8n на таком объёме упрётся в память и время выполнения, чистый Scrapy потребует писать оповещения и расписание самостоятельно.

## 8. Итоги и переход к лабораторной работе

Что студент должен унести с занятия:

1. Критерий перехода на фреймворк: тысячи страниц + обход по ссылкам + регулярность.
2. Путь запроса по компонентам Scrapy и понимание асинхронности `parse` (порядок страниц не гарантирован).
3. Паук с пагинацией и двухуровневым обходом, `cb_kwargs` для передачи контекста.
4. Item Pipeline из четырёх звеньев с учётом отброшенных записей в статистике.
5. Настройки вежливости (`ROBOTSTXT_OBEY`, задержки, параллелизм на домен, AutoThrottle) и предохранители (`CLOSESPIDER_*`, `DEPTH_LIMIT`).
6. `HTTPCACHE` для отладки и `JOBDIR` для возобновления.
7. Чтение статистики прогона как отчёта о качестве сбора.
8. Таблица соответствия Scrapy ↔ n8n и обоснование выбора по объёму и сложности.

**Лабораторная работа (6 часов):** разработать Scrapy-проект с двухуровневым обходом и пайплайном на 300+ записей, собрать данные в БД, продемонстрировать возобновление и вежливые настройки, реализовать эквивалентный n8n-workflow на подмножестве данных, сравнить подходы и оформить отчёт. Полное условие — в отдельном файле задания.

### Типичные ошибки студентов (предупредить заранее)

| Ошибка | Последствие | Что требовать |
|---|---|---|
| `ROBOTSTXT_OBEY = False` «чтобы работало» | нарушение правил сайта | `True` и объяснение, что именно запрещено |
| Нет `CLOSESPIDER_*` и `DEPTH_LIMIT` | обход всего сайта, тысячи лишних запросов | предохранители в учебных запусках |
| Вся логика в `parse`, без пайплайна | нечитаемый код, потеря контроля качества | вынести валидацию, очистку, дедупликацию |
| `DropItem` без учёта в статистике | молчаливая потеря данных | `stats.inc_value` на каждую причину |
| Опора на порядок обработки страниц | нестабильные результаты | передача контекста через `cb_kwargs` |
| Соединение с БД в `process_item` | десятки открытий соединения | `open_spider` / `close_spider`, пакетная запись |
| `time.sleep` внутри колбэка | блокировка цикла событий, потеря параллелизма | `DOWNLOAD_DELAY` и AutoThrottle |
| Отладка селекторов полными прогонами | лишняя нагрузка на сайт | `scrapy shell` и `HTTPCACHE_ENABLED` |
| Игнорирование статистики прогона | поломка селекторов замечается спустя недели | контроль `item_scraped_count` и доли пустых полей |